# Data

In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt -n 3

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [4]:
import collections
import numpy as np
import tqdm

def load(limit, path = "stand/log.local.txt"):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    with open(path) as f:
        req = list()
        reqid = None
        models = list()
        prevreqmodel = None
        reqmodel = dict()
        prevmodelid = -1
        bannermodelid = -1
        for i, line in tqdm.tqdm_notebook(enumerate(f)):
            if line.startswith("Model = 6;"):
                prevreqmodel = reqmodel
                reqmodel = dict()

            if line.startswith("Model = "):
                spl = line.split(" ")
                prevmodelid = int(spl[2][:-1])
                bannermodelid = max(bannermodelid , prevmodelid)
                reqmodel[prevmodelid] = readvector(spl[3])
            elif line.startswith("dbid"):
                spl = line.split(" ")
                dbid = int(spl[1][:-1])
                docembs[bannermodelid][dbid] = readvector(spl[2])
            elif line.startswith("seed"):
                if len(requests) >= limit:
                    break
                if req:
                    requests.append((reqid, prevreqmodel, sorted(req)))
                    req = list()
                reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
            else:
                req.append(
                    (int(line.split()[0]), float(line.split()[1]))
                )

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7):
            self.games_count = games_count
            self.key_games = np.random.choice(np.arange(games_count), key_size, replace=False)

            self.requests = requests
            np.random.shuffle(self.requests)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext()

In [5]:
ctx = load(1500)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
101 937 446


# Models

In [6]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in self.ctx.get_requests(t)
        ])
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):
        mses = list()
        results = list()
        for rec, tru in zip(self.recommend(t), self.get_user_scores(t)):
            mses.append((rec - tru) ** 2)
            rec = np.argsort(-rec)
            tru = np.argsort(-tru)
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            results.append(ev(rec[:topsize], tru[:topsize]) / float(topsize))
        print(np.mean(results), np.array(mses).mean(), len(results))
        return np.mean(results)

In [7]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "train_popbias": False,
    "train_bias": False,
    "verbose": True,
    "train_dssm": False,
    "loss": "mse"
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        self.fit_kwargs = fit_kwargs
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[2][i][1] for i in ctx.key_games])
                    for r_i in ctx.get_requests(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[2][g_i][1] for r in ctx.get_requests("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in ctx.get_requests("key")
        ])
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return self.embed_users[t]
    
    def get_game_embs(self):
        return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.train_bias = p["train_bias"]
        self.train_popbias = p["train_popbias"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        
        for i in range(p["n"]):
            def loss():
                def get_logits():
                    if self.train_dssm:
                        logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs))
                    else:
                        logits = train_user_embs @ self.W @ game_embs.T
                    
                    if self.train_popbias:
                        logits += self.pb * self.game_avg_scores["train"]     
                        
                    if self.train_bias:
                        logits += self.b

                    return logits
                        
                if self.loss == "mse":
                    logits = get_logits()
                    v = tf.reduce_mean((train_user_scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits = get_logits()
                    q_mean = train_user_scores.mean(axis=1)
                    # print(train_user_scores.shape, q_mean.shape)
                    v = tf.reduce_mean(((train_user_scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    loss_batch = 64 if "loss_batch" not in p else p["loss_batch"]
                    while True:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        if self.train_dssm:
                            logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs[game_slice]))
                        else:
                            logits = train_user_embs @ self.W @ game_embs[game_slice].T

                        if self.train_popbias:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                        if self.train_bias:
                            logits += self.b

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            train_user_scores[:, game_slice].astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    loss_batch = 64 if "loss_batch" not in p else p["loss_batch"]
                    game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                    if self.train_dssm:
                        logits = self.u_dssm(train_user_embs) @ tf.transpose(self.g_dssm(game_embs[game_slice]))
                    else:
                        logits = train_user_embs @ self.W @ game_embs[game_slice].T

                    if self.train_popbias:
                        logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_bias:
                        logits += self.b

                    scores = train_user_scores[:, game_slice]
                    # v = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
                    #    (scores.T > np.quantile(scores, 1 - 100./16000).T).T,
                    #    logits
                    #))
                    v = -tf.reduce_mean(tf.where(
                        (scores.T > np.quantile(scores, p["loss_q"]).T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                else:
                    assert False
                    
                v += tf.reduce_mean(self.W * self.W) * p["c"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            else:
                weights += [self.W]
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_bias:
                weights += [self.b]   
                
                
            opt.minimize(loss, weights).numpy()

    def recommend(self, t):
        if self.train_dssm:
            logits = self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))
        else:
            logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [ ]:
logs = list()

ev([Popular(ctx)], logs)

for n in [10, 100, 1000, 10000, 20000, 50000, 100000]:
    for train_dssm in [True]:
        c_grid = [0.1] if train_dssm else [0, 0.01, 0.1, 1, 10, 100, 1000]
        for c in c_grid:
            for train_popbias in [True]:
                for train_bias in [True]:
                    for loss in ["mse", "qmse", "ApproxNDCGLoss", "softmax"]:
                        lb_grid = [8, 16, 32, 64, 128] if loss in ["ApproxNDCGLoss", "softmax"] else [32]
                        for lb in lb_grid:
                            kw = {
                                "c": c,
                                "train_dssm": train_dssm,
                                "train_popbias": train_popbias,
                                "train_bias": train_bias,
                                "verbose": False,
                                "loss": loss,
                                "loss_batch": lb,
                                "loss_q": (lb - 1.5) / lb,
                                "n": n
                            }
                            for dssm_id in [6, 7, 8, 9]:
                                ev([
                                    DssmKnn(ctx, dssm_id, fit_kwargs=kw),
                                ], logs)
                            ev([
                                CBKnnV0(ctx, fit_kwargs=kw),
                            ], logs)




model =  <__main__.Popular object at 0x7f899c252048>
0.4702454642475987 1.4830163740223972 937
0.4759641255605382 1.6472645471376166 446
0.4702454642475987 0.4759641255605382
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10})
0.4727321237993597 1.4803604 937
0.4781390134529148 1.6447836 446
0.4727321237993597 0.4781390134529148
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'tra

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 10})
0.4621771611526148 1.4997734 937
0.4671076233183857 1.6634847 446
0.4621771611526148 0.4671076233183857
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 16, 'loss_q': 0.90625, 'n': 10})
0.46419423692636075 1.550676 937
0.469

self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 10})
0.4669370330843116 1.4981376 937
0.47181614349775786 1.6625999 446
0.4669370330843116 0.47181614349775786
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 10})
0.45889007470651016 1.5670952 937
0.4652242152466368 1.7305162 446
0.45889007470651016 0.4652242152466368
self.embed_users['train'].shape =  (937, 100

0.4689220917822839 1.497054 937
0.47381165919282514 1.6615623 446
0.4689220917822839 0.47381165919282514
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10})
0.4617502668089648 1.5239033 937
0.4675336322869955 1.6854494 446
0.4617502668089648 0.4675336322869955
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'sof

self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 100})
0.5389647812166489 1.3819258 937
0.5444170403587444 1.5582571 446
0.5389647812166489 0.5444170403587444
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 100})
0.5503308431163286 1.3716394 937
0.5556726457399103 1.5514507 446
0.5503308431163286 0.5556726457399103
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (1

...

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 1000})
0.6184311632870865 535.2198 937
0.6216591928251122 181.37292 446
0.6184311632870865 0.6216591928251122
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 8, 'loss_q': 0.8125, 'n': 1000})
0.28710779082177157 1355.4481 937
0.28854260

0.4923906083244397 6736.0845 937
0.49403587443946195 2567.9502 446
0.4923906083244397 0.49403587443946195
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 64, 'loss_q': 0.9765625, 'n': 1000})
0.4395197438633938 303.3607 937
0.43881165919282505 307.69907 446
0.4395197438633938 0.43881165919282505
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss':

self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'qmse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5547918890074707 0.16597636 937
0.5294618834080718 1.7078012 446
0.5547918890074707 0.5294618834080718
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'qmse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5564674493062967 0.14876276 937
0.5238116591928251 2.3295166 446
0.5564674493062967 0.5238116591928251
self.embed_users['train'].shape =  (937, 100)
self.embed_games.sha

0.5494130202774813 3.9281976 937
0.5507847533632286 3.9989977 446
0.5494130202774813 0.5507847533632286
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 10000})
0.5423372465314834 4.239975 937
0.5427578475336323 4.310461 446
0.5423372465314834 0.5427578475336323
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss

0.3666816143497758 1279.2572 446
0.36737459978655285 0.3666816143497758
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 8, 'loss_q': 0.8125, 'n': 10000})
0.40977588046958374 607.4854 937
0.4105156950672646 602.40967 446
0.40977588046958374 0.4105156950672646
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 8, 'loss_q'

In [12]:
sorted(logs, key = lambda x : -x[2])

[("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 50000})",
  0.7451227321237993,
  0.7195739910313902),
 ("DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 50000})",
  0.7340234791889008,
  0.706883408071749),
 ("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.7204375667022411,
  0.7056502242152466),
 ("DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.7128175026680896,
  0.698542600896861),
 ("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': Fals

In [14]:
logs[-1]

("DssmKnn(9,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'qmse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 100000})",
 0.5361045891141942,
 0.48860986547085206)

In [ ]:
for n in [100000]:
    for train_dssm in [True]:
        c_grid = [0.1] if train_dssm else [0, 0.01, 0.1, 1, 10, 100, 1000]
        for c in c_grid:
            for train_popbias in [True]:
                for train_bias in [True]:
                    for loss in ["mse", "qmse", "ApproxNDCGLoss", "softmax"]:
                        lb_grid = [8, 16, 32, 64, 128] if loss in ["ApproxNDCGLoss", "softmax"] else [32]
                        for lb in lb_grid:
                            kw = {
                                "c": c,
                                "train_dssm": train_dssm,
                                "train_popbias": train_popbias,
                                "train_bias": train_bias,
                                "verbose": False,
                                "loss": loss,
                                "loss_batch": lb,
                                "loss_q": (lb - 1.5) / lb,
                                "n": n
                            }
                            for dssm_id in [6, 7, 8, 9]:
                                ev([
                                    DssmKnn(ctx, dssm_id, fit_kwargs=kw),
                                ], logs)
                            ev([
                                CBKnnV0(ctx, fit_kwargs=kw),
                            ], logs)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)



model =  DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 100000})


In [18]:
logs[-1]

("DssmKnn(7,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'mse', 'loss_batch': 32, 'loss_q': 0.953125, 'n': 100000})",
 0.553948772678762,
 0.5106726457399103)

In [19]:
sorted(logs, key = lambda x : -x[2])

[("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 50000})",
  0.7451227321237993,
  0.7195739910313902),
 ("DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 50000})",
  0.7340234791889008,
  0.706883408071749),
 ("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.7204375667022411,
  0.7056502242152466),
 ("DssmKnn(8,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 20000})",
  0.7128175026680896,
  0.698542600896861),
 ("DssmKnn(6,{'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': True, 'verbose': Fals